# 🏥 Insurance Charges — Data Cleaning & Exploration

## Objective

Clean and explore a health insurance dataset to prepare it for actuarial analysis. The goal is to understand the key risk factors that drive insurance charges — age, BMI, smoking status, and region — before modeling premium pricing.

**Dataset:** 1,338 policyholders with demographic and health attributes.

**Source:** [Medical Cost Personal Dataset](https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

## 1. Load the Data

In [2]:
URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(URL)

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Shape: 1338 rows, 7 columns


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
# check data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


## 2. Data Quality Checks

Before any analysis we verify: missing values, duplicates, and outliers.

In [4]:
# missing values check
print("Missing values per column:")
print(df.isnull().sum())

# duplicate rows check
print(f"\nDuplicate rows: {df.duplicated().sum()}")

Missing values per column:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Duplicate rows: 1


In [5]:
# remove the duplicate row found above
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after removing duplicates: {df.shape}")

Shape after removing duplicates: (1337, 7)


In [6]:
# summary statistics for numeric columns
df.describe()

,age,bmi,children,charges
count,1337.000000,1337.000000,1337.000000,1337.000000
mean,39.222139,30.663452,1.095737,13279.121487
std,14.044333,6.100468,1.205571,12110.359656
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.290000,0.000000,4746.344000
50%,39.000000,30.400000,1.000000,9386.161300
75%,51.000000,34.700000,2.000000,16657.717450
max,64.000000,53.130000,5.000000,63770.428010


## 3. Feature Engineering

We create grouped categories that actuaries use for risk segmentation:
- **age_band**: groups age into actuarial bands
- **bmi_category**: standard WHO BMI classification
- **charge_level**: low / medium / high cost segments

In [7]:
# pd.cut splits a continuous column into labeled bands
df["age_band"] = pd.cut(
    df["age"],
    bins=[17, 29, 39, 49, 64],
    labels=["18-29", "30-39", "40-49", "50-64"]
)

# WHO BMI categories
df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)

# charge level tiers based on quartiles
df["charge_level"] = pd.cut(
    df["charges"],
    bins=[0, 5000, 15000, 100000],
    labels=["Low", "Medium", "High"]
)

df[["age", "age_band", "bmi", "bmi_category", "charges", "charge_level"]].head()

,age,age_band,bmi,bmi_category,charges,charge_level
0,19,18-29,27.900,Overweight,16884.92400,High
1,18,18-29,33.770,Obese,1725.55230,Low
2,28,18-29,33.000,Obese,4449.46200,Low
3,33,30-39,22.705,Normal,21984.47061,High
4,32,30-39,28.880,Overweight,3866.85520,Low


## 4. Exploratory Analysis — Key Risk Factors

In [8]:
# the single most important actuarial insight: smoker impact
smoker_avg = df.groupby("smoker")["charges"].mean().round(0)
multiplier = smoker_avg["yes"] / smoker_avg["no"]

print("Average charge by smoker status:")
print(smoker_avg)
print(f"\nSmokers cost {multiplier:.1f}x more than non-smokers")

Average charge by smoker status:
smoker
no      8441.0
yes    32050.0
Name: charges, dtype: float64

Smokers cost 3.8x more than non-smokers


In [9]:
# average charge across the main risk dimensions
print("Average charge by age band:")
print(df.groupby("age_band", observed=True)["charges"].mean().round(0))

print("\nAverage charge by BMI category:")
print(df.groupby("bmi_category", observed=True)["charges"].mean().round(0))

print("\nAverage charge by region:")
print(df.groupby("region", observed=True)["charges"].mean().round(0))

Average charge by age band:
age_band
18-29     9201.0
30-39    11739.0
40-49    14399.0
50-64    17903.0
Name: charges, dtype: float64

Average charge by BMI category:
bmi_category
Underweight     8658.0
Normal         10435.0
Overweight     10998.0
Obese          15581.0
Name: charges, dtype: float64

Average charge by region:
region
northeast    13406.0
northwest    12451.0
southeast    14735.0
southwest    12347.0
Name: charges, dtype: float64


In [10]:
# correlation between numeric factors and charges
numeric_cols = ["age", "bmi", "children", "charges"]
corr = df[numeric_cols].corr()["charges"].drop("charges").round(3)
print("Correlation with charges:")
print(corr.sort_values(ascending=False))

Correlation with charges:
age         0.298
bmi         0.198
children    0.067
Name: charges, dtype: float64


## 5. Save Cleaned Dataset

The cleaned dataset with engineered features is saved for the analysis scripts and Power BI.

In [11]:
df.to_csv("../data/insurance_clean.csv", index=False)
print(f"Saved: insurance_clean.csv ({df.shape[0]} rows, {df.shape[1]} columns)")

Saved: insurance_clean.csv (1337 rows, 10 columns)


## Key Takeaways

- **Smoking** is the dominant cost driver — smokers cost ~3.8x more
- **Age** shows a steady upward trend in charges across bands
- **Obesity** (BMI 30+) significantly increases average charges
- **Region** has a mild effect, with the Southeast being most expensive

These risk factors form the basis for the premium pricing model in the analysis scripts.